# Tutorial for running MS2Query and creating your own libraries

# optional: download a matchms cleaned library
The code below downloads an already matchms cleaned library and an MS2DeepScore model. You can also use your own library, but make sure you know what you are doing and clean the library first. If you just have a few reference spectra, it is probably best to combine your spectra with the reference spectra below to make sure the MS2Query search works properly. 

In [ ]:
import requests
import os
from tqdm import tqdm

def download_file(link, file_name):
    response = requests.get(link, stream=True)
    if os.path.exists(file_name):
        print(f"The file {file_name} already exists, the file won't be downloaded")
        return
    total_size = int(response.headers.get('content-length', 0))

    with open(file_name, "wb") as f, tqdm(desc="Downloading file", total=total_size, unit='B', unit_scale=True, unit_divisor=1024,) as bar:
        for chunk in response.iter_content(chunk_size=1024):
            if chunk:
                f.write(chunk)
                bar.update(len(chunk))  # Update progress bar by the chunk size
folder_to_store_zenodo_files = "./zenodo_files"
os.makedirs(folder_to_store_zenodo_files, exist_ok=True)

download_file("https://zenodo.org/records/16882111/files/merged_and_cleaned_libraries_1.mgf?download=1", 
              os.path.join(folder_to_store_zenodo_files, "merged_and_cleaned_libraries_1.mgf"))
download_file("https://zenodo.org/records/17826815/files/ms2deepscore_model.pt?download=1", 
              os.path.join(folder_to_store_zenodo_files, "ms2deepscore_model.pt"))

The file ./zenodo_files\data_split_inchikeys.json already exists, the file won't be downloaded
The file ./zenodo_files\merged_and_cleaned_libraries_1.mgf already exists, the file won't be downloaded
The file ./zenodo_files\ms2deepscore_model.pt already exists, the file won't be downloaded


# Specify file location 
Replace with your file names

In [ ]:
library_spectra_file = os.path.join(folder_to_store_zenodo_files, "merged_and_cleaned_libraries_1.mgf")
ms2deepscore_model_file_name = os.path.join(folder_to_store_zenodo_files, "ms2deepscore_model.pt")
query_spectrum_file = "replace_with_your_lib_spectra.mgf"

In [ ]:
from matchms.importing import load_from_mgf
from tqdm import tqdm

library_spectra = list(tqdm(load_from_mgf(library_spectra_file)))
query_spectra = list(tqdm(load_from_mgf(query_spectrum_file)))

1017531it [09:51, 1720.67it/s]


# Create the reference library files
The code below will precompute everything needed to run MS2Query. It will save this in the same folder as your ms2deepscore model. 
The files created are "embeddings.npz", "top_k_tanimoto_scores.parquet", "library_metadata.parquet". 

In [ ]:
from ms2query.ms2query_development.ReferenceLibrary import ReferenceLibrary
reference_library = ReferenceLibrary.create_from_spectra(library_spectra, ms2deepscore_model_file_name)

# Run MS2Query
The code above only has to be run once after that you can load the library faster from the saved files. 

In [ ]:
# no need to run if you just created the libary above
reference_library = ReferenceLibrary.load_from_directory(folder_to_store_zenodo_files)

In [ ]:
results = reference_library.run_ms2query(query_spectra)

print(results)

In [ ]:
results.to_csv("ms2query_results.csv")